# ECRTM avec UMAP

## Hypothèse
On applique UMAP **avant l'entraînement** d'ECRTM, directement sur les `word_embeddings`, afin de réduire la dimension de l'espace sémantique manipulé par la régularisation ECR.

## Choix expérimental
UMAP est placé avant l'entraînement et non après. Après apprentissage, il ne changerait ni la perte ECR ni le temps de calcul. Ici l'idée est de modifier l'espace géométrique sur lequel ECRTM apprend ses topics.


In [1]:
import sys
!{sys.executable} -m pip install -q pyyaml scipy pandas scikit-learn tqdm torch gensim umap-learn


In [2]:
import os
import time
import sys
import subprocess
from pathlib import Path
from types import SimpleNamespace

import yaml
import numpy as np
import pandas as pd
import scipy.io
import scipy.sparse as sp
import torch
import umap
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from sklearn.metrics import normalized_mutual_info_score
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

IS_KAGGLE = Path('/kaggle').exists()
MODEL_TAG = 'ECRTM_AVEC_UMAP'
FORCE_RECOMPUTE_METRICS = True
TARGET_DATASETS = ['20NG', 'AGNews', 'IMDB', 'YahooAnswer']
K_VALUES = [20, 50, 100]
SEED = 42
UMAP_CONFIG = {'enabled': True, 'n_components': 64, 'n_neighbors': 30, 'min_dist': 0.0, 'metric': 'cosine', 'random_state': SEED, 'normalize_before': True, 'normalize_after': True}


def _unique_paths(paths):
    uniq = []
    seen = set()
    for p in paths:
        try:
            rp = Path(p).resolve()
        except Exception:
            continue
        key = str(rp)
        if key not in seen:
            seen.add(key)
            uniq.append(rp)
    return uniq


def detect_project_root():
    if IS_KAGGLE:
        return Path('/kaggle/working')
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'ECTRM_source' / 'ECRTM' / 'data').exists():
            return candidate
        if (candidate / 'Projet_PPD' / 'Projet_PPD').exists():
            return candidate / 'Projet_PPD' / 'Projet_PPD'
        if (candidate / 'Projet_PPD').exists():
            return candidate / 'Projet_PPD'
    return cwd


PROJECT_ROOT = detect_project_root()
DATA_ROOT = PROJECT_ROOT / 'data'
OUT_DIR_ECRTM = (Path('/kaggle/working') / 'Sortie_mat' / 'output' / 'kaggle' / MODEL_TAG) if IS_KAGGLE else (PROJECT_ROOT / 'output' / MODEL_TAG)
OUT_DIR_ECRTM.mkdir(parents=True, exist_ok=True)


def detect_ecrtm_repo(project_root):
    cwd = Path.cwd().resolve()
    candidates = [project_root, project_root.parent, project_root / 'ECTRM_source' / 'ECRTM', project_root / 'ECTRM_source' / 'ECRTM' / 'ECRTM', project_root.parent / 'ECTRM_source' / 'ECRTM', project_root.parent / 'ECTRM_source' / 'ECRTM' / 'ECRTM', cwd / 'ECTRM_source' / 'ECRTM', cwd / 'ECTRM_source' / 'ECRTM' / 'ECRTM']
    if IS_KAGGLE:
        input_root = Path('/kaggle/input')
        if input_root.exists():
            for top in input_root.iterdir():
                if not top.is_dir():
                    continue
                candidates.extend([top / 'ECTRM_source' / 'ECRTM', top / 'ECTRM_source' / 'ECRTM' / 'ECRTM', top / 'PPD_2026_ECTRM' / 'ECTRM_source' / 'ECRTM', top / 'PPD_2026_ECTRM' / 'ECTRM_source' / 'ECRTM' / 'ECRTM', top / 'ECRTM'])
    for candidate in _unique_paths(candidates):
        if (candidate / 'models').exists() and (candidate / 'configs').exists():
            return candidate
    return None


def clone_ecrtm_repo_kaggle():
    if not IS_KAGGLE:
        return None
    clone_root = Path('/kaggle/working') / '_ecrtm_source'
    repo_dir = clone_root / 'ecrtm'
    clone_root.mkdir(parents=True, exist_ok=True)
    if not repo_dir.exists():
        try:
            subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/bobxwu/ecrtm', str(repo_dir)])
        except Exception:
            return None
    for candidate in [repo_dir / 'ECRTM', repo_dir]:
        if (candidate / 'models').exists() and (candidate / 'configs').exists():
            return candidate
    return None


ECRTM_REPO = detect_ecrtm_repo(PROJECT_ROOT)
if ECRTM_REPO is None:
    ECRTM_REPO = clone_ecrtm_repo_kaggle()
if ECRTM_REPO is None:
    raise FileNotFoundError('Impossible de localiser ECRTM (models/ + configs/).')
if str(ECRTM_REPO) not in sys.path:
    sys.path.append(str(ECRTM_REPO))
from models.ECRTM import ECRTM

CONFIGS = {'20NG': ECRTM_REPO / 'configs' / 'model' / 'ECRTM_20NG.yaml', 'AGNews': ECRTM_REPO / 'configs' / 'model' / 'ECRTM_AGNews.yaml', 'IMDB': ECRTM_REPO / 'configs' / 'model' / 'ECRTM_IMDB.yaml', 'YahooAnswer': ECRTM_REPO / 'configs' / 'model' / 'ECRTM_YahooAnswer.yaml'}
REQUIRED_DATA_FILES = ['train_bow.npz', 'test_bow.npz', 'train_labels.txt', 'test_labels.txt', 'vocab.txt', 'word_embeddings.npz']


def has_required_files(path: Path) -> bool:
    return path.exists() and path.is_dir() and all((path / f).exists() for f in REQUIRED_DATA_FILES)


def guess_dataset_name(path: Path):
    s = str(path).lower()
    if '20ng' in s or '20news' in s:
        return '20NG'
    if 'agnews' in s or 'ag_news' in s:
        return 'AGNews'
    if 'yahoo' in s:
        return 'YahooAnswer'
    if 'imdb' in s:
        return 'IMDB'
    return None


def iter_candidate_dirs(base: Path, max_depth: int = 6):
    if not base.exists():
        return
    base_depth = len(base.parts)
    for root, dirs, _ in os.walk(base):
        root_path = Path(root)
        depth = len(root_path.parts) - base_depth
        if depth > max_depth:
            dirs[:] = []
            continue
        yield root_path


def discover_dataset_dirs():
    found = {}
    for ds in TARGET_DATASETS:
        for p in [DATA_ROOT / ds, PROJECT_ROOT / 'ECTRM_source' / 'ECRTM' / 'data' / ds]:
            if has_required_files(p) and ds not in found:
                found[ds] = p
    scan_roots = [Path('/kaggle/input'), PROJECT_ROOT] if IS_KAGGLE else [PROJECT_ROOT]
    seen = set()
    for scan_root in scan_roots:
        for cand in iter_candidate_dirs(scan_root):
            cs = str(cand)
            if cs in seen:
                continue
            seen.add(cs)
            if not has_required_files(cand):
                continue
            ds = guess_dataset_name(cand)
            if ds is not None and ds not in found:
                found[ds] = cand
    return found


DATASET_DIRS = discover_dataset_dirs()
DATASETS = [ds for ds in TARGET_DATASETS if ds in DATASET_DIRS] or TARGET_DATASETS


2026-03-10 15:35:00.219035: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773156900.400272      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773156900.450630      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773156900.883940      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773156900.883982      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773156900.883985      55 computation_placer.cc:177] computation placer alr

In [3]:
def load_cfg(path):
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)


def load_vocab(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]


def load_bow_npz(path):
    X = sp.load_npz(path).astype(np.float32)
    return X.toarray()


def load_word_emb_npz(path):
    try:
        W = sp.load_npz(path)
        return W.astype(np.float32).toarray()
    except Exception:
        data = np.load(path)
        arr = data[list(data.keys())[0]] if isinstance(data, np.lib.npyio.NpzFile) else data
        return np.asarray(arr, dtype=np.float32)


In [4]:
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


DATA_CACHE = {}


def normalize_rows(arr, eps=1e-12):
    arr = np.asarray(arr, dtype=np.float32)
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    return arr / np.maximum(norms, eps)


def reduce_word_embeddings_with_umap(word_emb, dataset):
    word_emb = np.asarray(word_emb, dtype=np.float32)
    original_dim = int(word_emb.shape[1])
    if UMAP_CONFIG.get('normalize_before', True):
        word_emb = normalize_rows(word_emb)
    if (not UMAP_CONFIG['enabled']) or original_dim <= int(UMAP_CONFIG['n_components']):
        return word_emb.astype(np.float32), {'umap_enabled': False, 'umap_seconds': 0.0, 'embedding_dim_before': original_dim, 'embedding_dim_after': original_dim, 'umap_n_neighbors': 0}
    n_neighbors = min(int(UMAP_CONFIG['n_neighbors']), max(2, word_emb.shape[0] - 1))
    n_components = min(int(UMAP_CONFIG['n_components']), original_dim - 1 if original_dim > 2 else original_dim)
    if n_components < 2:
        return word_emb.astype(np.float32), {'umap_enabled': False, 'umap_seconds': 0.0, 'embedding_dim_before': original_dim, 'embedding_dim_after': original_dim, 'umap_n_neighbors': n_neighbors}
    start = time.perf_counter()
    reducer = umap.UMAP(n_components=n_components, n_neighbors=n_neighbors, min_dist=float(UMAP_CONFIG['min_dist']), metric=UMAP_CONFIG['metric'], random_state=int(UMAP_CONFIG['random_state']), transform_seed=int(UMAP_CONFIG['random_state']), low_memory=True)
    reduced = reducer.fit_transform(word_emb)
    if UMAP_CONFIG.get('normalize_after', True):
        reduced = normalize_rows(reduced)
    return np.asarray(reduced, dtype=np.float32), {'umap_enabled': True, 'umap_seconds': float(time.perf_counter() - start), 'embedding_dim_before': original_dim, 'embedding_dim_after': int(reduced.shape[1]), 'umap_n_neighbors': int(n_neighbors)}


def build_args_from_yaml(cfg, K, vocab_size, word_emb):
    cfg = dict(cfg)
    cfg['num_topic'] = K
    cfg['vocab_size'] = vocab_size
    cfg['word_embeddings'] = word_emb
    reduced_dim = int(word_emb.shape[1])
    for dim_key in ('embed_size', 'emb_size', 'embedding_dim', 'rho_size'):
        if dim_key in cfg:
            cfg[dim_key] = reduced_dim
    return SimpleNamespace(**cfg)


def infer_theta(model, bow_np, device, batch_size=256):
    model.eval()
    thetas = []
    with torch.no_grad():
        X = torch.from_numpy(bow_np).float()
        loader = DataLoader(TensorDataset(X), batch_size=batch_size, shuffle=False)
        for (bow,) in loader:
            bow = bow.to(device)
            theta = model.get_theta(bow)
            thetas.append(theta.cpu().numpy())
    return np.vstack(thetas)


def prepare_ecrtm_inputs(dataset):
    if dataset in DATA_CACHE:
        payload = dict(DATA_CACHE[dataset])
        payload['cache_hit'] = True
        return payload
    data_dir = DATASET_DIRS[dataset]
    cfg = load_cfg(CONFIGS[dataset])
    train_bow = load_bow_npz(data_dir / 'train_bow.npz')
    test_bow = load_bow_npz(data_dir / 'test_bow.npz')
    vocab = load_vocab(data_dir / 'vocab.txt')
    word_emb = load_word_emb_npz(data_dir / 'word_embeddings.npz')
    V = train_bow.shape[1]
    assert V == test_bow.shape[1] == len(vocab) == word_emb.shape[0], 'Mismatch vocab sizes'
    start = time.perf_counter()
    reduced_word_emb, umap_meta = reduce_word_embeddings_with_umap(word_emb, dataset)
    payload = {'cfg': cfg, 'train_bow': train_bow, 'test_bow': test_bow, 'word_emb': reduced_word_emb, 'prep_seconds': float(time.perf_counter() - start), 'umap_meta': umap_meta}
    DATA_CACHE[dataset] = payload
    payload = dict(payload)
    payload['cache_hit'] = False
    return payload


In [5]:
def train_one(dataset, K, seed=42):
    set_seed(seed)
    total_start = time.perf_counter()
    prepared = prepare_ecrtm_inputs(dataset)
    cfg = prepared['cfg']
    epochs = int(cfg['epochs'])
    batch_size = int(cfg['batch_size'])
    lr = float(cfg['learning_rate'])
    V = prepared['train_bow'].shape[1]
    args = build_args_from_yaml(cfg, K=K, vocab_size=V, word_emb=prepared['word_emb'])
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = ECRTM(args).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    X = torch.from_numpy(prepared['train_bow']).float()
    loader = DataLoader(TensorDataset(X), batch_size=batch_size, shuffle=True)
    train_start = time.perf_counter()
    epoch_bar = tqdm(range(1, epochs + 1), desc=f'{dataset} | K={K} | seed={seed} | UMAP')
    for epoch in epoch_bar:
        model.train()
        total_loss = 0.0
        for (bow,) in loader:
            bow = bow.to(device)
            rst = model(bow)
            loss = rst['loss']
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += float(loss.detach().cpu())
        epoch_bar.set_postfix(avg_loss=total_loss / max(len(loader), 1))
    train_seconds = time.perf_counter() - train_start
    model.eval()
    with torch.no_grad():
        beta = model.get_beta().cpu().numpy()
    train_theta = infer_theta(model, prepared['train_bow'], device=device, batch_size=256)
    test_theta = infer_theta(model, prepared['test_bow'], device=device, batch_size=256)
    out_dir = OUT_DIR_ECRTM / dataset
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f'{dataset}_{MODEL_TAG}_K{K}_seed{seed}.mat'
    scipy.io.savemat(str(out_path), {'beta': beta, 'train_theta': train_theta, 'test_theta': test_theta})
    prep_seconds = 0.0 if prepared['cache_hit'] else float(prepared['prep_seconds'])
    umap_seconds = 0.0 if prepared['cache_hit'] else float(prepared['umap_meta']['umap_seconds'])
    total_seconds = time.perf_counter() - total_start
    timing_row = {'model': MODEL_TAG, 'dataset': dataset, 'K': int(K), 'seed': int(seed), 'device': device, 'phase': 'train', 'prep_seconds': float(prep_seconds), 'umap_seconds': float(umap_seconds), 'train_seconds': float(train_seconds), 'total_seconds': float(total_seconds), 'prep_minutes': float(prep_seconds / 60.0), 'umap_minutes': float(umap_seconds / 60.0), 'train_minutes': float(train_seconds / 60.0), 'total_minutes': float(total_seconds / 60.0), 'embedding_dim_before': int(prepared['umap_meta']['embedding_dim_before']), 'embedding_dim_after': int(prepared['umap_meta']['embedding_dim_after']), 'umap_enabled': bool(prepared['umap_meta']['umap_enabled']), 'umap_n_neighbors': int(prepared['umap_meta']['umap_n_neighbors']), 'umap_cache_hit': bool(prepared['cache_hit'])}
    timing_path = out_dir / f'{dataset}_{MODEL_TAG}_K{K}_seed{seed}_timing.csv'
    pd.DataFrame([timing_row]).to_csv(timing_path, index=False)
    return {'mat_path': str(out_path), 'timing_path': str(timing_path), 'timing': timing_row}


## Fonctions de métriques


In [6]:
def purity_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    purity = 0
    for c in np.unique(y_pred):
        idx = np.where(y_pred == c)[0]
        if len(idx) > 0:
            _, counts = np.unique(y_true[idx], return_counts=True)
            purity += counts.max()
    return purity / max(len(y_true), 1)


_CACHE = {}


def build_tokenized_texts(train_bow, vocab, max_docs=10000):
    texts = []
    for doc in train_bow[:max_docs]:
        idx = np.where(doc > 0)[0]
        words = [vocab[i] for i in idx if i < len(vocab)]
        if len(words) >= 2:
            texts.append(words)
    return texts


def load_gensim_artifacts(dataset):
    if dataset in _CACHE:
        return _CACHE[dataset]

    if dataset not in DATASET_DIRS:
        raise KeyError(f'Dataset {dataset} introuvable dans DATASET_DIRS')
    data_dir = DATASET_DIRS[dataset]
    train_bow = load_bow_npz(data_dir / "train_bow.npz")
    vocab = load_vocab(data_dir / "vocab.txt")
    texts = build_tokenized_texts(train_bow, vocab)
    dictionary = Dictionary(texts)

    _CACHE[dataset] = (vocab, texts, dictionary)
    return _CACHE[dataset]


def compute_cv_gensim(beta, vocab, texts, dictionary, topn=15):
    topics = []
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:topn]
        topic_words = [
            vocab[i]
            for i in top_idx
            if i < len(vocab) and vocab[i] in dictionary.token2id
        ]
        if len(topic_words) >= 2:
            topics.append(topic_words)

    if not topics:
        return 0.0

    cm = CoherenceModel(
        topics=topics,
        texts=texts,
        dictionary=dictionary,
        coherence="c_v",
    )
    return float(cm.get_coherence())


## Pipeline des métriques


In [7]:
def process_ecrtm_dataset(dataset, K, seed=42):
    if dataset not in DATASET_DIRS:
        return None
    data_dir = DATASET_DIRS[dataset]
    dataset_dir = OUT_DIR_ECRTM / dataset
    dataset_dir.mkdir(parents=True, exist_ok=True)
    mat_path = dataset_dir / f'{dataset}_{MODEL_TAG}_K{K}_seed{seed}.mat'
    metrics_path = dataset_dir / f'metrics_{dataset}_K{K}_seed{seed}.csv'
    if metrics_path.exists() and not FORCE_RECOMPUTE_METRICS:
        return pd.read_csv(metrics_path).to_dict('records')[0]
    if not mat_path.exists():
        return None
    d = scipy.io.loadmat(str(mat_path))
    beta = np.asarray(d['beta'])
    if beta.shape[0] != K and beta.shape[1] == K:
        beta = beta.T
    theta_key = 'test_theta' if 'test_theta' in d else 'train_theta'
    theta = np.asarray(d.get(theta_key))
    if theta.ndim != 2:
        return None
    label_file = 'test_labels.txt' if theta_key == 'test_theta' else 'train_labels.txt'
    y_true = np.loadtxt(data_dir / label_file, dtype=int)
    y_pred = theta.argmax(axis=1)
    n = min(len(y_true), len(y_pred))
    y_true = y_true[:n]
    y_pred = y_pred[:n]
    purity = purity_score(y_true, y_pred)
    nmi = float(normalized_mutual_info_score(y_true, y_pred))
    vocab, texts, dictionary = load_gensim_artifacts(dataset)
    topics = []
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:25]
        topics.append([vocab[i] for i in top_idx if i < len(vocab)])
    all_words = [w for t in topics for w in t]
    td = len(set(all_words)) / max(len(all_words), 1)
    cv = compute_cv_gensim(beta, vocab, texts, dictionary, topn=15)
    res = {'Dataset': dataset, 'K': int(K), 'Seed': int(seed), 'C_V': round(cv, 4), 'TD': round(td, 4), 'Purity': round(float(purity), 4), 'NMI': round(float(nmi), 4)}
    pd.DataFrame([res]).to_csv(metrics_path, index=False)
    return res


## Exécution finale


In [8]:
training_time_rows = []
for dataset in DATASETS:
    if dataset not in DATASET_DIRS:
        continue
    for K in K_VALUES:
        result = train_one(dataset, K, seed=SEED)
        if isinstance(result, dict) and 'timing' in result:
            training_time_rows.append(result['timing'])
if training_time_rows:
    df_train_times = pd.DataFrame(training_time_rows).sort_values(['dataset', 'K']).reset_index(drop=True)
    display(df_train_times[['dataset', 'K', 'device', 'prep_seconds', 'umap_seconds', 'train_seconds', 'total_seconds', 'embedding_dim_before', 'embedding_dim_after', 'umap_cache_hit']])
    time_csv = OUT_DIR_ECRTM / f'{MODEL_TAG}_training_times.csv'
    df_train_times.to_csv(time_csv, index=False)


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


20NG | K=20 | seed=42 | UMAP:   0%|          | 0/500 [00:00<?, ?it/s]

20NG | K=50 | seed=42 | UMAP:   0%|          | 0/500 [00:00<?, ?it/s]

20NG | K=100 | seed=42 | UMAP:   0%|          | 0/500 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


...

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


YahooAnswer | K=20 | seed=42 | UMAP:   0%|          | 0/500 [00:00<?, ?it/s]

YahooAnswer | K=50 | seed=42 | UMAP:   0%|          | 0/500 [00:00<?, ?it/s]

YahooAnswer | K=100 | seed=42 | UMAP:   0%|          | 0/500 [00:00<?, ?it/s]

,dataset,K,device,prep_seconds,umap_seconds,train_seconds,total_seconds,embedding_dim_before,embedding_dim_after,umap_cache_hit
0,20NG,20,cuda,40.972577,40.970333,281.431617,323.741659,200,64,False
1,20NG,50,cuda,0.000000,0.000000,702.340489,702.652145,200,64,True
2,20NG,100,cuda,0.000000,0.000000,701.051743,701.340467,200,64,True
3,AGNews,20,cuda,21.908953,21.906719,617.746991,640.350298,200,64,False
4,AGNews,50,cuda,0.000000,0.000000,610.399893,610.580871,200,64,True
5,AGNews,100,cuda,0.000000,0.000000,615.484552,615.681282,200,64,True
6,IMDB,20,cuda,21.761442,21.759340,1384.648019,1408.714857,200,64,False
7,IMDB,50,cuda,0.000000,0.000000,1523.793999,1524.486286,200,64,True
8,IMDB,100,cuda,0.000000,0.000000,1521.336178,1522.040598,200,64,True
9,YahooAnswer,20,cuda,22.194035,22.192189,575.532113,598.427502,200,64,False


In [9]:
final_summary = []
for dataset in DATASETS:
    if dataset not in DATASET_DIRS:
        continue
    for K in K_VALUES:
        res = process_ecrtm_dataset(dataset, K, seed=SEED)
        if res is not None:
            final_summary.append(res)
if final_summary:
    df_results = pd.DataFrame(final_summary).sort_values(['Dataset', 'K'])
    display(df_results)
    out_csv = OUT_DIR_ECRTM / f'{MODEL_TAG}_ALL_METRICS.csv'
    df_results.to_csv(out_csv, index=False)


,Dataset,K,Seed,C_V,TD,Purity,NMI
0,20NG,20,42,0.6230,0.9320,0.5838,0.5591
1,20NG,50,42,0.5611,0.7056,0.6138,0.5374
2,20NG,100,42,0.4328,0.3964,0.5891,0.5087
3,AGNews,20,42,0.3803,0.9900,0.3132,0.0885
4,AGNews,50,42,0.4231,0.7136,0.7276,0.3230
5,AGNews,100,42,0.4229,0.3816,0.5748,0.2569
6,IMDB,20,42,0.3826,0.9280,0.7012,0.0748
7,IMDB,50,42,0.3701,0.6944,0.7630,0.0867
8,IMDB,100,42,0.3895,0.2636,0.6882,0.0474
9,YahooAnswer,20,42,0.4383,1.0000,0.5796,0.3413
